# Notebook for helping being updated on the latest information from study programs. 



In [1]:
import json
from utils.helpers import *

filepath = "config.json"

# Load the JSON file containing config
with open(filepath, "r", encoding="utf-8") as json_file:
    config_data = json.load(json_file)

### Scan the urls of config file and list the urls containing year (meaning schools holds different URLS for each year)

In [ ]:
print("Checking URLs containing year patterns:")
print("-" * 40)

schools_with_years = {}

# Iterate through schools and their study programs
for school in config_data:
    found_year = False
    year_programs = []
    study_programs = config_data[school]["study_programs"]
    
    for program, url in study_programs.items():
        # Look for year patterns like /2024h/, /2023/ etc in URL
        import re
        year_match = re.search(r'20\d{2}', url)
        
        if year_match:
            found_year = True
            year = year_match.group()
            year_programs.append(f"- {program}: Found year {year.strip('/')}")
            
            # Add to dictionary
            if school not in schools_with_years:
                schools_with_years[school] = {}
            schools_with_years[school][program] = {
                "url": url,
                "year": year
            }
    
    if found_year:
        print(f"\n{school}:")
        for program in year_programs:
            print(program)

### Find newest url for the studyprogram. 

In [ ]:
import requests
from datetime import datetime
current_year = datetime.now().year

#Loop over the schools that have years in their urls. 
for school in schools_with_years:
    #Skip kristiania and uit as they have a different structure. 
    if school == 'KRISTIANIA' or school == 'UIT':
        continue
    
    print(school)
    
    #Loop over the study programs for each school. 
    for program in schools_with_years[school]:
        
        print(f"\t{program}")
        
        url = schools_with_years[school][program]['url'] #URL for the study program. 
        
        year_match = re.search(r'20\d{2}', url) # Look for year patterns like /2024h/, /2023/ etc in URL
        
        if year_match:
            # Get the start and end positions of the year in the URL
            start = year_match.start()
            end = year_match.end()
            
            year_in_original_url = url[start:end] #year used in the url. 
            
            updated_url = url[:start] + str(current_year) + url[end:] # Replace the year portion with the current year. 
            
            print(f"\n\t\tOriginal URL: {url} \n\t\tUpdated URL: {updated_url} \n")
            
            #ping the url to check if it is working with the current year. 
            response = requests.get(updated_url)
            
            if response.status_code == 200:
                url_is_working = True
                print(f" \t\tURL is working: {updated_url}") #URL is returning 200
            else:
                url_is_working = False
                print(f" \t\tURL is not working: {updated_url}") #URL is not returning 200
            
            if url_is_working:
                print("\t\t", "---------"*10)
                print(f"\t\tThere exists a similar working url using the current year ({current_year}). NB! Please confirm its content before considering updating it. ")
                if year_in_original_url != str(current_year):
                    print(f"\t\tThe year in the original url is {year_in_original_url}. This is different from the current year ({current_year}).")
                    print(f"\t\tThere is a possibility for changing the URL to {current_year}.")
                else:
                    print(f"\t\tThe year in the original url is {year_in_original_url}. This is the same as the current year ({current_year}).")
                    print(f"\t\tYou are up to date!")
                    
            else:
                print("\t\t", "---------"*10)
                print(f"\t\tThere does not exist a similar working url using the current year ({current_year}).")
                print(f"\t\tYou are up to date!")
        
        print("\n\n")